# 02b — APSIM Next Generation (Maize), sem calibração, como gerador de variáveis

Segundo modelo de processos do estudo, ao lado do MZA-FAO (notebook 02), para testar se um modelo
de cultura mais detalhado agrega informação ao modelo híbrido. APSIM NG compilado do código aberto
(`APSIMInitiative/ApsimX`, .NET 8), a partir do exemplo oficial `Maize.apsimx`:

- clima diário BR-DWGD (notebook 01); vapor d'água pela UR e pela temperatura média;
- perfil físico de 7 camadas (0–1,8 m): granulometria do MapBiomas Solo (Coleção 3) por camada →
  LL15, DUL e SAT por Saxton & Rawls (2006); água disponível escalada para a AWC do município (01b);
- cultivar genérica `A_100`, 6 plantas m⁻², 200 kg N ha⁻¹ na semeadura (N não limitante);
- semeadura todo ano na data do cenário (9 datas, 5/15/25 de jan–mar), 1999–2025, simulação contínua
  (o solo evolui entre safras em pousio).

**Sem calibração**: o rendimento do APSIM não é usado como previsão; as variáveis do ciclo (rendimento
potencial simulado, número de grãos, estresse hídrico por período, Tmin nas fases) alimentam o ML.
Uma simulação por município × data (≈ 3.600), ~2,5 h em 4 núcleos. Retoma de onde parou.

In [1]:
import shutil
import tempfile
from pathlib import Path

import pandas as pd

import modelo_apsim as A

RAIZ = Path.cwd().parent
DADOS, RESULTADOS = RAIZ / "dados", RAIZ / "resultados"
PARTES = RESULTADOS / "apsim_partes"
PARTES.mkdir(exist_ok=True)
DATAS = [(m, d) for m in (1, 2, 3) for d in (5, 15, 25)]

mun = pd.read_csv(DADOS / "municipios_pr.csv").set_index("cod_ibge")
camadas = pd.read_csv(DADOS / "solo_camadas_pr.csv")
feitos = {int(p.stem) for p in PARTES.glob("*.parquet")}
print(f"{len(feitos)} de {len(mun)} municípios já simulados")

0 de 399 municípios já simulados


In [2]:
falhas = {}
for cod, m in mun.iterrows():
    if cod in feitos:
        continue
    pasta = Path(tempfile.mkdtemp(prefix="apsim_"))
    try:
        clima = pd.read_parquet(DADOS / "xavier_diario_pr.parquet", filters=[("cod_ibge", "==", cod)])
        A.escrever_met(clima.set_index("data").sort_index(), m["latitude"], pasta / "clima.met")
        solo = A.perfil(camadas[camadas["cod_ibge"] == cod], m["dta_mm_m"])
        A.montar(cod, pasta / "clima.met", solo, DATAS, pasta / "sim.apsimx")
        A.resumir(A.rodar(pasta / "sim.apsimx", cpus=4), cod).to_parquet(PARTES / f"{cod}.parquet", index=False)
    except Exception as erro:
        falhas[cod] = str(erro)[:300]
    finally:
        shutil.rmtree(pasta, ignore_errors=True)
print("falhas:", len(falhas), list(falhas.items())[:3])

falhas: 0 []


In [3]:
apsim = pd.concat([pd.read_parquet(p) for p in sorted(PARTES.glob("*.parquet"))], ignore_index=True)
apsim.to_parquet(RESULTADOS / "simulacao_apsim_pr.parquet", index=False)
ordem = [f"{d:02d}/{m:02d}" for m, d in DATAS]
print(apsim.shape, apsim["cod_ibge"].nunique(), "municípios")
apsim.groupby("semeadura").mean(numeric_only=True).drop(columns=["cod_ibge", "ano"]).loc[ordem].round(2).T

(96957, 16) 399 municípios


semeadura,05/01,15/01,25/01,05/02,15/02,25/02,05/03,15/03,25/03
ap_rend,7726.79,7324.13,7023.99,6597.85,6135.83,5462.86,4988.88,4213.53,3307.38
ap_biomassa,12203.34,11610.27,11225.55,10639.75,10122.04,9308.50,8810.65,8001.89,6860.44
ap_n_graos,3216.14,3116.15,2967.33,2824.57,2671.68,2425.26,2236.38,1968.06,1601.25
ap_iaf_max,2.73,2.61,2.55,2.47,2.41,2.29,2.23,2.18,2.04
ap_maturou,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
ap_dias_floracao,50.01,49.91,50.06,50.76,51.63,53.08,55.16,58.60,63.19
ap_ciclo_dias,99.31,101.50,104.96,110.25,115.68,121.43,126.21,131.52,136.11
ap_estresse_veg,0.61,0.65,0.55,0.77,0.66,0.76,0.79,0.87,1.33
ap_tmin_veg,16.00,15.97,15.73,15.24,14.35,12.97,11.38,9.31,7.25
ap_estresse_flor,0.22,0.24,0.19,0.21,0.24,0.20,0.14,0.14,0.11
